## Notebook 8: CBA

In [1]:
# Repo-root discovery + import of cityheat
import sys
from pathlib import Path

def _find_root():
    start = Path.cwd()
    for cand in [start, *start.parents]:
        if (cand / "cityheat").is_dir() and (cand / "configs").is_dir():
            return cand
    raise RuntimeError("Could not find repo root with 'cityheat' and 'configs' folders.")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityheat.nbsetup import bootstrap
C = bootstrap("Rome")  # switch to "Barcelona" later

PROJECT, CITY, CFG, cfg = C["PROJECT"], C["CITY"], C["CFG"], C["cfg"]
OUT, INT, BASE = C["OUT"], C["INT"], C["BASE"]
p_LCZ, p_UrbClim, p_GVI, p_CoolEff = C["p_LCZ"], C["p_UrbClim"], C["p_GVI"], C["p_CoolEff"]

print("Repo root:", ROOT)
print("✔ Loaded config:", CFG)
print("→ Outputs:", OUT)
print("→ Data base:", BASE)

Repo root: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat
✔ Loaded config: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/configs/rome.yml
→ Outputs: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/Rome
→ Data base: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/Rome


In [ ]:
from pathlib import Path
import json, gdown
from datetime import datetime

manifest = ROOT / "data_manifests" / "rome_gdrive.json"
base_dir = Path(BASE)
M = json.loads(Path(manifest).read_text())

def sync_from_gdrive(M, base_dir, use_cookies=True, force=False):
    base_dir = Path(base_dir)
    for e in M["entries"]:
        dest = base_dir / e["dest"]
        dest.parent.mkdir(parents=True, exist_ok=True)

        if e["type"] == "folder":
            marker = dest / ".synced.ok"
            needs = force or (not dest.exists()) or (not any(dest.iterdir())) or (not marker.exists())
            if needs:
                gdown.download_folder(id=e["id"], output=str(dest), quiet=False, use_cookies=use_cookies)
                marker.write_text(json.dumps({"id": e["id"], "when": datetime.now().isoformat()}))

        elif e["type"] == "file":
            sidecar = dest.with_suffix(dest.suffix + ".synced")
            needs = force or (not dest.exists()) or (not sidecar.exists())
            if needs:
                gdown.download(id=e["id"], output=str(dest), quiet=False, use_cookies=use_cookies)
                sidecar.write_text(json.dumps({"id": e["id"], "size": dest.stat().st_size, "when": datetime.now().isoformat()}))

# run once per notebook
sync_from_gdrive(M, base_dir)

# sanity checks
checks = [
    base_dir/"gviRome/gvi_Rome.csv",
    base_dir/"UrbClim/UrbClimWBGTmax/WBGT_year_daily_max_2014.nc",
    base_dir/"CAPZONE/CAPZONE.shp",
]
for p in checks:
    print(p, "OK" if p.exists() else "MISSING")

**Loading from before**

In [2]:
# loading everything needed for the CBA 
import json, numpy as np, pandas as pd, geopandas as gpd, rasterio as rio
import scipy.sparse as sp
from pathlib import Path
from climada.hazard import Hazard
from climada.entity import Exposures

INT, OUT = Path(INT), Path(OUT)

def _exists(p):
    p = Path(p)
    if not p.exists():
        print(f"[WARN] Missing: {p}")
        return False
    return True

# from Notebook 2
tmpl_path = INT / "template_ref.tif"
mask_tif  = INT / "city_mask.tif"
age_npz   = INT / "age_on_ref.npz"
pop_npz   = INT / "pop_on_ref.npz"
fua_gp    = OUT / "rome_fua.gpkg"

# loading ref grid + mask
with rio.open(tmpl_path) as src:
    ref_meta = src.meta.copy()
    HGT, WDT = ref_meta["height"], ref_meta["width"]

CITY_MASK = np.load(INT / "city_mask.npz")["city_mask"].astype(bool)

with rio.open(mask_tif) as src:
    CITY_MASK = src.read(1).astype(bool)

ref_data = np.where(CITY_MASK, 0.0, np.nan)
HAZ_MASK = np.isfinite(ref_data)

Z = np.load(INT / "age_on_ref.npz")
age_on_ref = {
    "<15":   Z["lt15"],
    "15-64": Z["a15_64"],
    "65+":   Z["g65"],
}
pop_on_ref = np.load(INT / "pop_on_ref.npz")["pop"]


muni_on_ref = None
if _exists(OUT / "muni_on_ref.tif"):
    with rio.open(OUT / "muni_on_ref.tif") as src:
        muni_on_ref = src.read(1).astype(np.int32)

cap_on_ref = None
if _exists(INT / "cap_on_ref.npz"):
    cap_on_ref = np.load(INT / "cap_on_ref.npz")["cap_on_ref"].astype(np.int32)

# exposure
pop_on_ref = np.load(INT / "pop_on_ref.npz")["pop"].astype(float)
ages_npz   = np.load(INT / "age_on_ref.npz")
AGE_LT15   = ages_npz["lt15"].astype(float)
AGE_15_64  = ages_npz["a15_64"].astype(float)
AGE_65P    = ages_npz["g65"].astype(float)

# CLIMADA exposures 
exp = Exposures.from_hdf5(INT / "exposure_people.h5")

# Hazards 
H          = Hazard.from_hdf5(INT / "hazard_HEAT.h5")
H_trees_dd = Hazard.from_hdf5(INT / "hazard_HEAT_trees_dd.h5")
assert np.all(H.event_id == H_trees_dd.event_id), "Event IDs differ between H and H_trees_dd."

f_ref = None
aux_npz = INT / "trees_policy_artifacts.npz"
if aux_npz.exists():
    z = np.load(aux_npz)
    f_ref = z.get("f_ref", None)

# Impact functions (frozen, age-specific) 
with open(INT / "if_curves_frozen.json", "r") as fh:
    IF_FREEZE = json.load(fh)  # keys: "<15", "15-64", "65+" with arrays 'intensity','mdd'

# AC coverage (baseline vs policy)
ac_pack = np.load(INT / "ac_coverage_maps.npz")
coverage_base   = ac_pack["coverage_base"].astype(float)
coverage_policy = ac_pack["coverage_policy"].astype(float)
CAP_MASK        = ac_pack["CAP_MASK"].astype(bool) if "CAP_MASK" in ac_pack else CITY_MASK

# Admin vectors/tables for costs and reporting 
muni = None
if _exists(OUT / "rome_regions.gpkg"):
    try:
        muni = gpd.read_file(OUT / "rome_regions.gpkg", layer="muni")
    except Exception as e:
        print("[WARN] Could not read 'muni' layer:", e)

trees_tbl    = pd.read_csv(OUT / "trees_tbl.csv")      if _exists(OUT / "trees_tbl.csv")    else None
muni_cov     = pd.read_csv(OUT / "muni_cov.csv")       if _exists(OUT / "muni_cov.csv")     else None
muni_tbl_all = pd.read_csv(OUT / "muni_tbl_all.csv")   if _exists(OUT / "muni_tbl_all.csv") else None

# checks 
sum_dGVI = float(trees_tbl["dGVI"].clip(lower=0).sum()) if trees_tbl is not None else np.nan
print("\n[NB8 · Loaded]")
print("  Grid:", pop_on_ref.shape, "| CITY_MASK:", CITY_MASK.shape)
print("  Ages:", AGE_LT15.shape, AGE_15_64.shape, AGE_65P.shape)
print("  Hazards: events =", len(H.event_id))
print("  AC maps:", coverage_base.shape, coverage_policy.shape)
print("  Municipi raster:", None if muni_on_ref is None else muni_on_ref.shape)
print("  Tables:",
      "trees_tbl✓" if trees_tbl is not None else "trees_tbl×", "|",
      "muni_cov✓" if muni_cov is not None else "muni_cov×", "|",
      "muni_tbl_all✓" if muni_tbl_all is not None else "muni_tbl_all×")
if trees_tbl is not None:
    print(f"  ΣΔGVI (0–1) = {sum_dGVI:.6f}  |  index-points (1–100) = {sum_dGVI*100:.4f}")

# hazard columns match grid size?
try:
    cols = H.intensity.shape[1]
    grid_n = coverage_base.size
    print(f"  Sanity H.cols==grid? {cols == grid_n}  (cols={cols}, grid_n={grid_n})")
except Exception:
    pass


[NB8 · Loaded]
  Grid: (266, 353) | CITY_MASK: (266, 353)
  Ages: (266, 353) (266, 353) (266, 353)
  Hazards: events = 5
  AC maps: (266, 353) (266, 353)
  Municipi raster: (266, 353)
  Tables: trees_tbl✓ | muni_cov✓ | muni_tbl_all✓
  ΣΔGVI (0–1) = 0.298932  |  index-points (1–100) = 29.8932
  Sanity H.cols==grid? True  (cols=93898, grid_n=93898)


**Trees: parametrisation of costs**

In [3]:
# Trees CAPEX and O&M 
R = 0.03 # discount rate 
T = 25   # time horizon 

# How much greening we need? Increase in GVI? 
# DELTA_INDEX: city-wide GVI increase required by the policy, on the 1–100 scale
DELTA_INDEX = float(trees_tbl['dGVI'].clip(lower=0).sum()) * 100.0  # 29.8932
# annual_index: spreads total evenly over 25 y (linear): how many index points we add each year
annual_index = DELTA_INDEX / T                                      # linear ramp (index-pts per year)
print(f"Delta Index: {DELTA_INDEX:,.0f}")

# Cost per index point (investment)
# One index-point of GVI costs €10M to build
# This is an investment at the time of planting, not yearly
CAPEX_PER_INDEX_PT = 10_000_000.0 

# total one-off investment requirement 
# undiscounted total investment needed to reach the target if paid it all “upfront"
TREES_CAPEX_T0 = CAPEX_PER_INDEX_PT * DELTA_INDEX  # €298.9M
print(f"Trees — Total investment requirement (undiscounted): €{TREES_CAPEX_T0:,.0f}")

# “discounted average” CAPEX per year (cf formula article)
# not used to build the NPV 
# just a reporting figure 
EAC_capex = TREES_CAPEX_T0 / ((1 + R)**T * T)  # €5.7M/yr 
print(f"Trees — EAC (CAPEX): €{EAC_capex:,.0f}/yr")

# phased investment NPV if plant linearly 
# not everything paid today, payment schedule implied by linearity: every year, buy annual_index points, and pay 10eurM per point. 
# each year's spend discounted and summed to get the CAPEX NPV
# CAPEX number we need to combine with O&M for a program total
def npv_capex_linear(delta_index_total, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT):
    inc = delta_index_total / years
    npv = 0.0
    for t in range(years):               # t = 0..years-1
        capex_t = capex_per_index * inc  # spend this year for this year's increment
        npv += capex_t / ((1 + r) ** t)
    return float(npv)

PV_trees_capex = npv_capex_linear(DELTA_INDEX, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT)
print(f"Trees — NPV CAPEX (linear phasing): €{PV_trees_capex:,.0f}")

# Calibrate O&M 
# Converting 27eur per year into eur per index point per year
# => O&M scales with same index-point unit used for CAPEX
# if 210 eur buys one's tree worth of index, same index costs 27eur per year tto maintain
CAPEX_PER_TREE = 210.0   # € (REGREEN median)
OM_PER_TREE_YR = 27.0    # €/yr
OM_PER_INDEX_PT_YR = (OM_PER_TREE_YR / CAPEX_PER_TREE) * CAPEX_PER_INDEX_PT  
print(f"OM per index point year: M€/y{OM_PER_INDEX_PT_YR:,.0f}")

# we don't have to put 25 even if our T is 25, for ex, with 30, CAP would never be hit
LIFETIME_YEARS = 25  

# O&M as overlapping cohorts, discounted to NPV 
# each year we add a planting cohort (size = annual_index). Every cohort adds same annual O&M in all subsequent years. 
# In year 10, 11 active cohorts so O&M that year is 11* per cohort amount
# Discounting each year's O&M and summing: O&M NPV
def npv_om_cohorts(delta_index_total, years=T, r=R,
                   om_per_index_per_year=OM_PER_INDEX_PT_YR, lifetime=LIFETIME_YEARS):
    """
    O&M: each year adds 'inc' index pts; each cohort pays the same O&M every year after planting,
    up to 'lifetime' years. Sum all cohorts active in each year; discount to NPV.
    """
    inc = delta_index_total / years
    npv = 0.0
    for t in range(years):  # year t cashflow
        active = t + 1
        if lifetime is not None:
            active = min(active, lifetime)
        om_t = active * om_per_index_per_year * inc
        npv += om_t / ((1 + r) ** t)
    return float(npv)

PV_trees_om = npv_om_cohorts(DELTA_INDEX, years=T, r=R,
                             om_per_index_per_year=OM_PER_INDEX_PT_YR,
                             lifetime=LIFETIME_YEARS)
print(f"Trees — NPV O&M (cohorts): €{PV_trees_om:,.0f}")

# Program totals
# Phased CAPEX NPV and cohort O&M NPV: program NPV for trees
PV_trees_total = PV_trees_capex + PV_trees_om
print(f"Trees — NPV total (CAPEX phased + O&M cohorts): €{PV_trees_total:,.0f}")

# OPTIONAL
# to show per year number for O&M or total, dividing their NPVs by standard annuity factor 
# just for presentation if needed. 
AF = (1 - (1 + R)**(-T)) / R
EAC_om_optional   = PV_trees_om / AF
EAC_total_optional = PV_trees_total / AF
print(f"(Optional) Trees — EAC (O&M only): €{EAC_om_optional:,.0f}/yr")
print(f"(Optional) Trees — EAC (total):   €{EAC_total_optional:,.0f}/yr")

Delta Index: 30
Trees — Total investment requirement (undiscounted): €298,932,306
Trees — EAC (CAPEX): €5,710,869/yr
Trees — NPV CAPEX (linear phasing): €214,460,519
OM per index point year: M€/y1,285,714
Trees — NPV O&M (cohorts): €316,454,777
Trees — NPV total (CAPEX phased + O&M cohorts): €530,915,296
(Optional) Trees — EAC (O&M only): €18,173,324/yr
(Optional) Trees — EAC (total):   €30,489,335/yr


**Sensitivity paved streets**

TO DO IF IT'S RIGHT, WITH 5379.0 instead of 210.

**Cost AC**

In [4]:
# AC costs: per-user accounting 
R = 0.03 # discount rate
T = 25   # horizon
AC_CAPEX_PER_USER   = 500.0           # CAPEX per AC user (before I was using units by saying 2.2 persons/AC)
AC_MAINT_RATE       = 0.05            # % of CAPEX per year, per user
AC_LIFETIME_YEARS   = 10              # replacement cycle, years
TARIFF_EUR_PER_KWH  = 0.25

# taking constant yearly amount and discounts each year 1...T, and summing them
# using this for annual maintenance and annual electricity
def pv_level_flow(annual, r=R, T=T):
    yrs = np.arange(1, T+1, dtype=float)
    return float(np.sum(annual * (1+r)**(-yrs)))


# summing the discounted purchases at t=0, life, 2*life,... <= T
# models buying and replacing AC for all added users
def pv_replacements(n_items, capex_per_item, r=R, T=T, life=AC_LIFETIME_YEARS):
    """NPV of replacing 'n_items' every 'life' years within horizon T, first purchase at t=0."""
    pv = 0.0
    t = 0
    while t <= T:
        pv += n_items * capex_per_item / ((1+r)**t if t > 0 else 1.0)
        t += life
    return pv

# merging muni_cov (base, policy AC shares and pop by Municipio) with muni_tbl_all  (per AC user kWh)
# dropping outside municipi rows (muni_id >0)
# df dshare: how much the share of AC users rises per Municipio
df = (muni_cov.merge(muni_tbl_all[['muni_id','kwh_per_user_muni']], on='muni_id', how='left')
               .fillna({'kwh_per_user_muni': 0.0}))
df = df.loc[df['muni_id'] > 0].copy()
df['dshare'] = (df['ac_policy_muni'] - df['ac_base_muni']).clip(lower=0.0)

# pop * increase in AC share, summed across Municipi
# total new AC users created by the policy
added_users       = float(np.nansum(df['pop_muni'] * df['dshare']))

# Annual O&M and electricity, per user
# per user, every year : 0.05*500
maint_per_user_yr = AC_MAINT_RATE * AC_CAPEX_PER_USER
# per-user kWh × added users (pop × dshare) summed across municipi
# so per-AC-user (not per capita)* change in AC share (policy-baseline)*pop*tariff
# sum over areas
# per AC user kWh * how many users we added * price per kWh sulmmed across Municipi
# citywide annual electricity bill due to added users
elec_per_user_yr  = float(np.nansum(df['pop_muni'] * df['dshare'] * df['kwh_per_user_muni'])) * TARIFF_EUR_PER_KWH

# PV : discount to present value
# buying for everyone at t=0
# replace at years 10 and 20
# each discounted
PV_ac_capex  = pv_replacements(added_users, AC_CAPEX_PER_USER, r=R, T=T, life=AC_LIFETIME_YEARS)
# maintenance is the same every year for all added users, discounting
PV_ac_maint  = pv_level_flow(added_users * maint_per_user_yr, r=R, T=T)
# annual electricity (citywide) discounted
PV_ac_elec   = pv_level_flow(elec_per_user_yr, r=R, T=T)
# sum of everything
PV_ac_total  = PV_ac_capex + PV_ac_maint + PV_ac_elec

print(f"AC — added users ≈ {added_users:,.0f}")
print(f"AC — PV capex €{PV_ac_capex:,.0f} | PV maint €{PV_ac_maint:,.0f} | PV electricity €{PV_ac_elec:,.0f}")
print(f"AC — PV total €{PV_ac_total:,.0f}")

# optional
# standard annuity factor: PV/AF
AF = (1 - (1 + R)**(-T)) / R
EAC_ac_total = PV_ac_total / AF
print(f"(Optional) AC — EAC (total): €{EAC_ac_total:,.0f}/yr")

AC — added users ≈ 189,443
AC — PV capex €217,648,572 | PV maint €82,470,118 | PV electricity €533,571,193
AC — PV total €833,689,883
(Optional) AC — EAC (total): €47,877,035/yr


In [5]:
AF = (1 - (1+R)**(-T)) / R
elec_eur_per_yr_check = float(np.nansum(df['pop_muni'] * df['dshare'] * df['kwh_per_user_muni'])) * TARIFF_EUR_PER_KWH
print("Annual elec € (calc):", elec_eur_per_yr_check)
print("PV elec € (AF×annual):", elec_eur_per_yr_check * AF, "vs", PV_ac_elec)
print("Annual maint € (calc):", added_users * AC_MAINT_RATE * AC_CAPEX_PER_USER)
print("PV maint € (AF×annual):", (added_users * AC_MAINT_RATE * AC_CAPEX_PER_USER) * AF, "vs", PV_ac_maint)
pv_capex_per_user = AC_CAPEX_PER_USER * (1 + (1+R)**-AC_LIFETIME_YEARS + (1+R)**-(2*AC_LIFETIME_YEARS))
print("PV capex per user (closed form):", pv_capex_per_user, "→ total:", pv_capex_per_user * added_users, "vs", PV_ac_capex)

Annual elec € (calc): 30641857.67277193
PV elec € (AF×annual): 533571193.19109845 vs 533571193.19109803
Annual maint € (calc): 4736083.316627184
PV maint € (AF×annual): 82470118.27062704 vs 82470118.27062696
PV capex per user (closed form): 1148.8848345415297 → total: 217648571.90392485 vs 217648571.90392488


**Benefits and summary**

In [6]:
# I had to do this otherwise I had a problem with my hazards...

In [7]:
import numpy as np
import pandas as pd

def ensure_event_names(H, prefix="HEAT"):
    n = len(H.event_id)
    names = getattr(H, "event_name", None)
    need = (names is None) or (len(names) != n)

    if need:
        if hasattr(H, "date") and (H.date is not None) and (len(H.date) == n):
            years = [pd.to_datetime(d).year for d in H.date]
            H.event_name = np.asarray([f"{prefix}_{y}" for y in years], dtype=object)
        else:
            H.event_name = np.asarray([f"{prefix}_{eid}" for eid in H.event_id], dtype=object)
    return H

# fixing both hazards and keeping names aligned
H          = ensure_event_names(H, prefix="HEAT")
H_trees_dd = ensure_event_names(H_trees_dd, prefix="HEAT")

# forcing identical ordering/info across the pair
assert np.all(H.event_id == H_trees_dd.event_id), "Event IDs differ between H and H_trees_dd."
H_trees_dd.event_name = np.array(H.event_name, dtype=object)

# checks
print("events:", len(H.event_id))
print("names  (H):", len(H.event_name))
print("names  (H_trees_dd):", len(H_trees_dd.event_name))

events: 5
names  (H): 5
names  (H_trees_dd): 5


In [8]:
H.write_hdf5(INT / "hazard_HEAT.h5")
H_trees_dd.write_hdf5(INT / "hazard_HEAT_trees_dd.h5")

In [9]:
import numpy as np
import pandas as pd

def _infer_years_from(H):
    """Best-effort: try date→year, else event_name, else event_id."""
    n = len(H.event_id)

    # from date
    if getattr(H, "date", None) is not None and len(H.date) == n:
        try:
            yrs = [pd.to_datetime(d).year for d in H.date]
            return yrs
        except Exception:
            pass

    # from event_name like 'HEAT_2030'
    nm = getattr(H, "event_name", None)
    if nm is not None and len(nm) == n:
        yrs = []
        for s in nm:
            y = None
            for token in str(s).split("_"):
                if token.isdigit() and 1900 <= int(token) <= 2200:
                    y = int(token); break
            yrs.append(y)
        if all(y is not None for y in yrs):
            return yrs

    # from event_id if they look like years
    yrs = []
    for i, eid in enumerate(H.event_id):
        y = int(eid) if (str(eid).isdigit() and 1900 <= int(eid) <= 2200) else (2000 + i)
        yrs.append(y)
    return yrs

def ensure_event_meta(H, prefix="HEAT"):
    """Make event_name, date, frequency consistent with event_id length."""
    n = len(H.event_id)

    # names
    names = getattr(H, "event_name", None)
    if names is None or len(names) != n:
        yrs = _infer_years_from(H)
        H.event_name = np.asarray([f"{prefix}_{y}" for y in yrs], dtype=object)

    # dates
    dates = getattr(H, "date", None)
    if dates is None or len(dates) != n:
        yrs = _infer_years_from(H)
        # pick a mid-summer placeholder date for each event year
        H.date = np.asarray([pd.Timestamp(f"{y}-07-01") for y in yrs], dtype="datetime64[ns]")

    # frequency
    freq = getattr(H, "frequency", None)
    if freq is None or len(freq) != n:
        H.frequency = np.ones(n, dtype=float)
        H.frequency_unit = "1/year"  # safe default

    return H

# apply to both hazards and hard-align their meta 
H          = ensure_event_meta(H, prefix="HEAT")
H_trees_dd = ensure_event_meta(H_trees_dd, prefix="HEAT")

# enforce identical ordering/meta across the pair
assert np.all(H.event_id == H_trees_dd.event_id), "Event IDs differ between H and H_trees_dd."
H_trees_dd.event_name    = np.array(H.event_name,    dtype=object)
H_trees_dd.date          = np.array(H.date,          dtype="datetime64[ns]")
H_trees_dd.frequency     = np.array(H.frequency,     dtype=float)
H_trees_dd.frequency_unit = H.frequency_unit

print("OK:",
      len(H.event_id), len(H.event_name), len(H.date), len(H.frequency), "|",
      len(H_trees_dd.event_id), len(H_trees_dd.event_name), len(H_trees_dd.date), len(H_trees_dd.frequency))

OK: 5 5 5 5 | 5 5 5 5


In [10]:
# BENEFITS (PV avoided deaths) 
import numpy as np, pandas as pd
from copy import deepcopy
from climada.entity.impact_funcs import ImpactFunc, ImpactFuncSet
from climada.engine import ImpactCalc

# param
HORIZON_YEARS = 25
DISCOUNT_RATE = 0.03
TREE_RAMP_YEARS = 12 # tree ramp: linearly grows benefits over 12 y until maturity of trees
tree_ramp = np.minimum(np.arange(1, HORIZON_YEARS+1)/TREE_RAMP_YEARS, 1.0)

# age-varying AC efficacy (fractional reduction in MDD at 100% coverage)
EFF_BUCKETS = {"<15":0.20, "15-64":0.30, "65+":0.40}
EFF_UNIFORM = 0.30  

# helpers 
# tales a yearly series (ex year 1...T) and discounts each year back to present, then sums
def pv_of_stream(cashflows, r=DISCOUNT_RATE):
    yrs = np.arange(1, len(cashflows)+1, dtype=float)
    return float(np.sum(np.asarray(cashflows, float) * (1+r)**(-yrs)))

# impact values only for a few event years. Function fills gaps by linear interpolation, extends or truncates to 25 years
def annualize_to_horizon(series, horizon_years):
    """Interpolate event-year series to yearly length horizon_years."""
    yrs = np.array(series.index, dtype=int)
    s = series.reindex(range(yrs[0], yrs[-1]+1)).interpolate().ffill().bfill()
    if len(s) < horizon_years:
        tail = pd.Series([s.iloc[-1]]*(horizon_years - len(s)),
                         index=range(s.index[-1]+1, s.index[-1] + (horizon_years-len(s)) + 1))
        s = pd.concat([s, tail])
    else:
        s = s.iloc[:horizon_years]
    return s.to_numpy(float)

# weighted average with population weights
def popw_mean(x, pop, mask=None):
    m = np.isfinite(x) & np.isfinite(pop) & (pop > 0)
    if mask is not None: m &= mask
    if not np.any(m): return np.nan
    return float(np.nansum(x[m]*pop[m]) / np.nansum(pop[m]))

# impact_series and benef_seris: turn CLIMADA"'s per event impacts into Pandas series indexed by event year
# benef=baseline-policy
def impact_series(impact_obj, H):
    vals = np.ravel(impact_obj.at_event).astype(float)
    yrs  = [int(y) for y in H.event_id.tolist()]
    return pd.Series(vals, index=yrs)

def benefits_series(impact_base, impact_pol, H):
    base = impact_series(impact_base, H)
    pol  = impact_series(impact_pol,  H)
    ser  = base - pol
    return ser, float(ser.mean())

# rebuilding age-specific impact functions but scales their severity (MDD) down according to AC coverage and efficacy assumptions
def ifset_from_freeze_scaled(IF_FREEZE, age_scale_dict):
    """Rebuild ImpactFuncSet from frozen curves, scaling MDD per age."""
    funcs = []
    for age, d in IF_FREEZE.items():
        f = ImpactFunc(haz_type="HEAT", id=int(d["id"]))
        f.intensity = np.asarray(d["intensity"], float)
        f.mdd       = np.clip(np.asarray(d["mdd"], float) * float(age_scale_dict.get(age,1.0)), 0.0, 1.0)
        # keep paa if present; else 1s
        paa = d.get("paa", None)
        f.paa = np.asarray(paa, float) if paa is not None else np.ones_like(f.intensity, float)
        f.name = f"{age} × {age_scale_dict.get(age,1.0):.3f}"
        f.check()
        funcs.append(f)
    s = ImpactFuncSet(funcs); s.check()
    return s

# coverage => age multipliers (MDD attenuation), pop-weighted over CAP universe 
# computing pop weighted average AC coverage under baseline and policy
# for each age group, multiplier : 1-e*c : shrinks MDD curve
# => 2 ImpactFuncSets: one for baseline AC and one for policy AC
mask_use = CAP_MASK if 'CAP_MASK' in globals() else np.isfinite(pop_on_ref)
base_cov_mean   = popw_mean(coverage_base,   pop_on_ref, mask_use)
policy_cov_mean = popw_mean(coverage_policy, pop_on_ref, mask_use)

m_base_age   = {age: (1.0 - float(EFF_BUCKETS.get(age, EFF_UNIFORM)) * float(base_cov_mean))   for age in EFF_BUCKETS}
m_policy_age = {age: (1.0 - float(EFF_BUCKETS.get(age, EFF_UNIFORM)) * float(policy_cov_mean)) for age in EFF_BUCKETS}

age_ifs_base = ifset_from_freeze_scaled(IF_FREEZE, m_base_age)    # baseline AC
age_ifs_pol  = ifset_from_freeze_scaled(IF_FREEZE, m_policy_age)  # policy AC

# impacts (city-wide deaths per event) 
impact_base_AC   = ImpactCalc(exp, age_ifs_base, H).impact()         # baseline AC, baseline hazard
impact_trees_AC  = ImpactCalc(exp, age_ifs_base, H_trees_dd).impact()# baseline AC, trees hazard
impact_ac_pol    = ImpactCalc(exp, age_ifs_pol,  H).impact()         # policy AC, baseline hazard
impact_both_pol  = ImpactCalc(exp, age_ifs_pol,  H_trees_dd).impact()# policy AC, trees hazard

# computing avoided deaths series for Trees, AC, Both by substracting policy from baseline 
# event-year avoided deaths series 
trees_ser, _ = benefits_series(impact_base_AC, impact_trees_AC, H)
ac_ser,    _ = benefits_series(impact_base_AC, impact_ac_pol,   H)
both_ser,  _ = benefits_series(impact_base_AC, impact_both_pol, H)

# converting each sparse series to yearly over 25 y
# + 12 y growth ramp to trees component (trees take time to deliver full cooling)
# for both: AC's annual series (instant effect) and add only trees component and ramp that. Avoids double counting.
trees_yr = annualize_to_horizon(trees_ser, HORIZON_YEARS) * tree_ramp
ac_yr    = annualize_to_horizon(ac_ser,    HORIZON_YEARS)
# isolate the trees component inside "both" and ramp only that part
both_yr  = ac_yr + annualize_to_horizon(both_ser - ac_ser, HORIZON_YEARS) * tree_ramp

# discounting to present value
PV_b_tree = pv_of_stream(trees_yr, DISCOUNT_RATE)
PV_b_ac   = pv_of_stream(ac_yr,    DISCOUNT_RATE)
PV_b_both = pv_of_stream(both_yr,  DISCOUNT_RATE)

print(f"PV benefits (avoided deaths) — Trees: {PV_b_tree:,.2f} | AC: {PV_b_ac:,.2f} | Both: {PV_b_both:,.2f}")

# per-event table 
benefits_df = pd.DataFrame({
    "Year": trees_ser.index,
    "avoided_Trees": trees_ser.values,
    "avoided_AC": ac_ser.values,
    "avoided_Both": both_ser.values
})
benefits_df

PV benefits (avoided deaths) — Trees: 2,284.70 | AC: 982.67 | Both: 3,168.44


,Year,avoided_Trees,avoided_AC,avoided_Both
0,2030,116.246559,38.616767,149.838224
1,2050,212.326135,71.812569,274.940689
2,2070,394.480930,135.216017,512.569975
3,2090,640.890886,221.717846,834.737215
4,2100,745.395330,259.294459,972.256961


**Summary**

In [11]:
import numpy as np
import pandas as pd

def safe_ratio(c, b): 
    return float(c/b) if b and b > 1e-9 else np.inf

PV_c_trees   = float(PV_trees_total)
PV_c_ac      = float(PV_ac_total)

PV_b_tree_px = float(PV_b_tree)
PV_b_ac_px   = float(PV_b_ac)
PV_b_both_px = float(PV_b_both)

summary_main = pd.DataFrame([
    {
        "Policy": "Trees only",
        "PV_cost_eur": PV_c_trees,
        "PV_avoided_deaths": PV_b_tree_px,
        "Cost_per_avoided_death_eur": safe_ratio(PV_c_trees, PV_b_tree_px),
        "trees_CAPEX_EAC_eur_per_yr": float(EAC_capex),              
        "trees_O&M_EAC_optional_eur_per_yr": float(EAC_om_optional),            
        "added_AC_users": 0.0
    },
    {
        "Policy": "AC only",
        "PV_cost_eur": PV_c_ac,
        "PV_avoided_deaths": PV_b_ac_px,
        "Cost_per_avoided_death_eur": safe_ratio(PV_c_ac, PV_b_ac_px),
        "trees_CAPEX_EAC_eur_per_yr": 0.0,
        "trees_O&M_EAC_optional_eur_per_yr": 0.0,
        "added_AC_users": float(added_users)
    },
    {
        "Policy": "Both (vs baseline)",
        "PV_cost_eur": PV_c_trees + PV_c_ac,
        "PV_avoided_deaths": PV_b_both_px,
        "Cost_per_avoided_death_eur": safe_ratio(PV_c_trees + PV_c_ac, PV_b_both_px),
        "trees_CAPEX_EAC_eur_per_yr": float(EAC_capex),
        "trees_O&M_EAC_optional_eur_per_yr": float(EAC_om_optional),
        "added_AC_users": float(added_users)
    },
]).round(2)

display(summary_main)

,Policy,PV_cost_eur,PV_avoided_deaths,Cost_per_avoided_death_eur,trees_CAPEX_EAC_eur_per_yr,trees_O&M_EAC_optional_eur_per_yr,added_AC_users
0,Trees only,5.309153e+08,2284.70,232378.25,5710869.37,18173324.14,0.00
1,AC only,8.336899e+08,982.67,848388.80,0.00,0.00,189443.33
2,Both (vs baseline),1.364605e+09,3168.44,430686.51,5710869.37,18173324.14,189443.33


**On a PV budget**

**Senstivity**